# March ML Mania 2026 - Mega Ensemble
**Combines predictions from all notebooks into the ultimate submission.**

This notebook reads submission CSVs from the other notebooks and creates
an optimally weighted mega-ensemble.

Prerequisites: Run these notebooks first and save their outputs as Kaggle datasets:
1. train_kaggle.ipynb -> submission.csv
2. advanced_models_kaggle.ipynb -> submission_stage2_advanced.csv
3. expert_models_kaggle.ipynb -> submission_stage2_expert.csv
4. foundation_models_kaggle.ipynb -> submission_stage2_temporal.csv

OR: Paste all model code into a single notebook (see standalone mode below).

## 0. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists("/kaggle/input")
CLIP_MIN, CLIP_MAX = 0.05, 0.95
SEED = 42
np.random.seed(SEED)

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")
    OUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(__file__).parent.parent / "data" / "raw"
    OUT_DIR = Path(__file__).parent.parent / ".tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Competition Data

In [ ]:
m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
sub1 = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['SeedNum'] = w_seeds['Seed'].str[1:3].astype(int)
all_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)
print(f"Stage 1: {len(sub1)} matchups, Stage 2: {len(sub2)} matchups")

## 2. Standalone Mode: Full Pipeline
This section builds all models from scratch so the notebook is self-contained.
Each model section generates OOF predictions for validation and final predictions.

In [ ]:
# ========== CORE INFRASTRUCTURE ==========
print("Building core infrastructure...")

m_reg_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
w_reg_compact = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")

# Elo System
class EloSystem:
    def __init__(self, k=32, home_adv=100, margin_mult=0.006, reversion=0.25):
        self.k, self.home_adv, self.margin_mult, self.reversion = k, home_adv, margin_mult, reversion
        self.ratings, self.initial = {}, 1500
    def get(self, t): return self.ratings.get(t, self.initial)
    def expected(self, ra, rb): return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))
    def update(self, w, l, margin, wloc='N'):
        rw, rl = self.get(w), self.get(l)
        rw_a = rw + (self.home_adv if wloc == 'H' else 0)
        rl_a = rl + (self.home_adv if wloc == 'A' else 0)
        exp_w = self.expected(rw_a, rl_a)
        mov = np.log(abs(margin) + 1) * (2.2 / (abs(rw - rl) * self.margin_mult + 2.2))
        adj = self.k * mov * (1 - exp_w)
        self.ratings[w], self.ratings[l] = rw + adj, rl - adj
    def new_season(self):
        for t in self.ratings:
            self.ratings[t] = self.ratings[t] * (1 - self.reversion) + self.initial * self.reversion

def build_elo(reg_df, tourney_df=None, k=32):
    elo = EloSystem(k=k)
    all_g = pd.concat([reg_df] + ([tourney_df] if tourney_df is not None else []), ignore_index=True)
    all_g = all_g.sort_values(['Season', 'DayNum']).reset_index(drop=True)
    season_ratings, prev = {}, None
    for _, g in all_g.iterrows():
        if g['Season'] != prev:
            if prev is not None: elo.new_season()
            prev = g['Season']
        elo.update(g['WTeamID'], g['LTeamID'], g['WScore'] - g['LScore'], g.get('WLoc', 'N'))
        if 132 <= g['DayNum'] <= 133:
            season_ratings[g['Season']] = dict(elo.ratings)
    season_ratings[all_g['Season'].max()] = dict(elo.ratings)
    rows = [{'Season': s, 'TeamID': t, 'EloRating': r} for s, rats in season_ratings.items() for t, r in rats.items()]
    return pd.DataFrame(rows), elo

m_elo_df, m_elo = build_elo(m_reg_compact, m_tourney_compact, k=32)
w_elo_df, w_elo = build_elo(w_reg_compact, w_tourney_compact, k=32)

# Team Stats
def compute_team_stats(det_df):
    def extract(df, p):
        o = 'L' if p == 'W' else 'W'
        return pd.DataFrame({'Season': df['Season'], 'TeamID': df[f'{p}TeamID'], 'DayNum': df['DayNum'],
            'Win': 1 if p == 'W' else 0, 'Score': df[f'{p}Score'], 'OppScore': df[f'{o}Score'],
            'FGM': df[f'{p}FGM'], 'FGA': df[f'{p}FGA'], 'FGM3': df[f'{p}FGM3'], 'FGA3': df[f'{p}FGA3'],
            'FTM': df[f'{p}FTM'], 'FTA': df[f'{p}FTA'], 'OR': df[f'{p}OR'], 'DR': df[f'{p}DR'],
            'Ast': df[f'{p}Ast'], 'TO': df[f'{p}TO'], 'Stl': df[f'{p}Stl'], 'Blk': df[f'{p}Blk'],
            'OppOR': df[f'{o}OR'], 'OppDR': df[f'{o}DR'], 'OppFGA': df[f'{o}FGA'],
            'OppFTA': df[f'{o}FTA'], 'OppTO': df[f'{o}TO'], 'OppFGM': df[f'{o}FGM'],
            'OppFGM3': df[f'{o}FGM3']})
    all_g = pd.concat([extract(det_df, 'W'), extract(det_df, 'L')], ignore_index=True)
    reg = all_g[all_g['DayNum'] < 132]
    agg = reg.groupby(['Season', 'TeamID']).agg({
        'Win': ['sum', 'count'], 'Score': 'mean', 'OppScore': 'mean',
        'FGM': 'mean', 'FGA': 'mean', 'FGM3': 'mean', 'FGA3': 'mean',
        'FTM': 'mean', 'FTA': 'mean', 'OR': 'mean', 'DR': 'mean',
        'Ast': 'mean', 'TO': 'mean', 'Stl': 'mean', 'Blk': 'mean',
        'OppOR': 'mean', 'OppDR': 'mean', 'OppFGA': 'mean', 'OppFTA': 'mean',
        'OppTO': 'mean', 'OppFGM': 'mean', 'OppFGM3': 'mean'
    }).reset_index()
    agg.columns = ['Season', 'TeamID', 'Wins', 'Games', 'Score', 'OppScore',
                    'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR',
                    'Ast', 'TO', 'Stl', 'Blk', 'OppOR', 'OppDR', 'OppFGA',
                    'OppFTA', 'OppTO', 'OppFGM', 'OppFGM3']
    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['PointDiff'] = agg['Score'] - agg['OppScore']
    agg['eFG_pct'] = (agg['FGM'] + 0.5 * agg['FGM3']) / agg['FGA']
    poss = agg['FGA'] + 0.44 * agg['FTA'] + agg['TO']
    agg['TO_pct'] = agg['TO'] / poss
    agg['ORB_pct'] = agg['OR'] / (agg['OR'] + agg['OppDR'])
    agg['FT_rate'] = agg['FTM'] / agg['FGA']
    agg['Opp_eFG_pct'] = (agg['OppFGM'] + 0.5 * agg['OppFGM3']) / agg['OppFGA']
    opp_poss = agg['OppFGA'] + 0.44 * agg['OppFTA'] + agg['OppTO']
    agg['OffRating'] = agg['Score'] / poss * 100
    agg['DefRating'] = agg['OppScore'] / opp_poss * 100
    agg['NetRating'] = agg['OffRating'] - agg['DefRating']
    agg['Pace'] = (poss + opp_poss) / 2
    agg['FG3_pct'] = agg['FGM3'] / agg['FGA3']
    agg['FT_pct'] = agg['FTM'] / agg['FTA']
    agg['Ast_TO'] = agg['Ast'] / agg['TO']
    # Last 10
    l10 = reg.sort_values('DayNum').groupby(['Season', 'TeamID']).tail(10)
    l10a = l10.groupby(['Season', 'TeamID']).agg({'Win': 'mean', 'Score': 'mean', 'OppScore': 'mean'}).reset_index()
    l10a.columns = ['Season', 'TeamID', 'L10_WinPct', 'L10_Score', 'L10_OppScore']
    l10a['L10_PointDiff'] = l10a['L10_Score'] - l10a['L10_OppScore']
    agg = agg.merge(l10a, on=['Season', 'TeamID'], how='left')
    gm = reg.copy(); gm['Margin'] = gm['Score'] - gm['OppScore']
    cons = gm.groupby(['Season', 'TeamID'])['Margin'].std().reset_index(name='MarginStd')
    agg = agg.merge(cons, on=['Season', 'TeamID'], how='left')
    return agg

m_stats = compute_team_stats(m_reg_detailed)
w_stats = compute_team_stats(w_reg_detailed)

# Massey
TOP_SYS = ['POM', 'SAG', 'MOR', 'DOL', 'COL', 'RPI']
eos = m_massey[(m_massey['RankingDayNum'] >= 128) & (m_massey['RankingDayNum'] <= 133) &
               (m_massey['SystemName'].isin(TOP_SYS))]
eos = eos.sort_values('RankingDayNum').groupby(['Season', 'SystemName', 'TeamID']).tail(1)
m_massey_feat = eos.pivot_table(index=['Season', 'TeamID'], columns='SystemName',
                                 values='OrdinalRank', aggfunc='first').reset_index()
rank_cols = [c for c in m_massey_feat.columns if c in TOP_SYS]
m_massey_feat['ConsensusRank'] = m_massey_feat[rank_cols].mean(axis=1)

# Feature vector
TEAM_FEATURES = ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
                  'OffRating', 'DefRating', 'NetRating', 'Pace', 'FG3_pct', 'FT_pct',
                  'Ast_TO', 'Opp_eFG_pct', 'L10_WinPct', 'L10_PointDiff', 'MarginStd',
                  'Score', 'OppScore', 'Stl', 'Blk']

def get_team_vector(stats_df, elo_df, season, team_id):
    row = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_id)]
    if len(row) == 0: return None
    r = row.iloc[0]
    feats = [r.get(f, 0) for f in TEAM_FEATURES]
    elo_row = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_id)]
    feats.append(elo_row.iloc[0]['EloRating'] if len(elo_row) > 0 else 1500)
    return np.array(feats, dtype=np.float32)

N_TEAM_FEATURES = len(TEAM_FEATURES) + 1
print(f"Core infrastructure ready. {N_TEAM_FEATURES} team features.")

In [ ]:
# ========== BUILD TRAINING DATA ==========
def build_training_data(tourney_df, seeds_df, stats_df, elo_df, massey_df=None):
    rows, targets, meta_rows = [], [], []
    for _, game in tourney_df.iterrows():
        season = game['Season']
        w_id, l_id = game['WTeamID'], game['LTeamID']
        team_a, team_b = min(w_id, l_id), max(w_id, l_id)
        target = 1 if team_a == w_id else 0
        vec_a = get_team_vector(stats_df, elo_df, season, team_a)
        vec_b = get_team_vector(stats_df, elo_df, season, team_b)
        if vec_a is None or vec_b is None: continue
        sa = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        sb = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        if len(sa) == 0 or len(sb) == 0: continue
        seed_a, seed_b = sa.iloc[0]['SeedNum'], sb.iloc[0]['SeedNum']
        diff = vec_a - vec_b
        tab = list(diff) + [seed_a - seed_b, seed_a, seed_b]
        # Massey
        if massey_df is not None:
            am = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_a)]
            bm = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_b)]
            for sys_name in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
                if len(am) > 0 and len(bm) > 0 and sys_name in am.columns:
                    va = am.iloc[0][sys_name] if not pd.isna(am.iloc[0].get(sys_name)) else 150
                    vb = bm.iloc[0][sys_name] if not pd.isna(bm.iloc[0].get(sys_name)) else 150
                    tab.append(va - vb)
                else:
                    tab.append(0)
        # Interactions
        elo_diff = vec_a[-1] - vec_b[-1]
        net_diff = diff[TEAM_FEATURES.index('NetRating')]
        tab.extend([(seed_a - seed_b) * elo_diff, (seed_a - seed_b) * net_diff])
        rows.append(np.array(tab, dtype=np.float32))
        targets.append(target)
        meta_rows.append({'Season': season, 'TeamA': team_a, 'TeamB': team_b,
                          'SeedA': seed_a, 'SeedB': seed_b})
    return np.array(rows, dtype=np.float32), np.array(targets, dtype=np.float32), pd.DataFrame(meta_rows)

print("Building training data...")
m_tab, m_y, m_meta = build_training_data(m_tourney_compact, m_seeds, m_stats, m_elo_df, m_massey_feat)
w_tab, w_y, w_meta = build_training_data(w_tourney_compact, w_seeds, w_stats, w_elo_df)

tab_all = np.vstack([m_tab, np.pad(w_tab, ((0,0),(0, m_tab.shape[1] - w_tab.shape[1])))])
y_all = np.concatenate([m_y, w_y])
meta_all = pd.concat([m_meta, w_meta], ignore_index=True)
seasons_all = meta_all['Season'].values
tab_all = np.nan_to_num(tab_all, nan=0.0)

tab_scaler = StandardScaler()
tab_scaled = tab_scaler.fit_transform(tab_all)
N_TAB = tab_all.shape[1]
print(f"Training data: {len(y_all)} games, {N_TAB} features")

## 3. Train All Diverse Models for Mega-Ensemble

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier, BayesianRidge

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

def cv_model(model_fn, X, y, seasons, name, use_scaled=False):
    """Generic leave-one-season-out CV."""
    data = tab_scaled if use_scaled else X
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = model_fn()
        m.fit(data[tr], y[tr])
        if hasattr(m, 'predict_proba'):
            oof[va] = np.clip(m.predict_proba(data[va])[:, 1], CLIP_MIN, CLIP_MAX)
        else:
            oof[va] = np.clip(m.predict(data[va]), CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name:30s}: Brier={bs:.4f}")
    return oof, bs

In [ ]:
print("=" * 60)
print("TRAINING ALL MODELS")
print("=" * 60)

all_oof = {}
all_brier = {}

# 1. Logistic Regressions
oof, bs = cv_model(lambda: LogisticRegression(C=0.5, max_iter=1000, random_state=SEED),
                    tab_all, y_all, seasons_all, "LR (C=0.5)", use_scaled=True)
all_oof['LR-0.5'], all_brier['LR-0.5'] = oof, bs

oof, bs = cv_model(lambda: LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
                    tab_all, y_all, seasons_all, "LR (C=1.0)", use_scaled=True)
all_oof['LR-1.0'], all_brier['LR-1.0'] = oof, bs

oof, bs = cv_model(lambda: LogisticRegression(C=0.1, max_iter=1000, random_state=SEED),
                    tab_all, y_all, seasons_all, "LR (C=0.1)", use_scaled=True)
all_oof['LR-0.1'], all_brier['LR-0.1'] = oof, bs

# 2. Elastic Net
oof, bs = cv_model(lambda: SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-4,
                                           l1_ratio=0.5, max_iter=1000, random_state=SEED),
                    tab_all, y_all, seasons_all, "ElasticNet", use_scaled=True)
all_oof['ElasticNet'], all_brier['ElasticNet'] = oof, bs

# 3. Bayesian Ridge
oof, bs = cv_model(lambda: BayesianRidge(max_iter=300),
                    tab_all, y_all, seasons_all, "BayesianRidge", use_scaled=True)
all_oof['BayesianRidge'], all_brier['BayesianRidge'] = oof, bs

# 4. SVM
oof, bs = cv_model(lambda: SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=SEED),
                    tab_all, y_all, seasons_all, "SVM-RBF", use_scaled=True)
all_oof['SVM-RBF'], all_brier['SVM-RBF'] = oof, bs

# 5. KNN
oof, bs = cv_model(lambda: KNeighborsClassifier(n_neighbors=50, weights='distance'),
                    tab_all, y_all, seasons_all, "KNN-50", use_scaled=True)
all_oof['KNN'], all_brier['KNN'] = oof, bs

# 6. Random Forest
oof, bs = cv_model(lambda: RandomForestClassifier(n_estimators=500, max_depth=8,
                    min_samples_leaf=10, max_features='sqrt', random_state=SEED, n_jobs=-1),
                    tab_all, y_all, seasons_all, "RandomForest")
all_oof['RF'], all_brier['RF'] = oof, bs

# 7. Extra Trees
oof, bs = cv_model(lambda: ExtraTreesClassifier(n_estimators=500, max_depth=10,
                    min_samples_leaf=8, max_features='sqrt', random_state=SEED, n_jobs=-1),
                    tab_all, y_all, seasons_all, "ExtraTrees")
all_oof['ET'], all_brier['ET'] = oof, bs

# 8. Gradient Boosting (sklearn)
oof, bs = cv_model(lambda: GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                    max_depth=4, subsample=0.8, min_samples_leaf=10, random_state=SEED),
                    tab_all, y_all, seasons_all, "GradientBoosting")
all_oof['GBC'], all_brier['GBC'] = oof, bs

# 9. Sklearn MLP
oof, bs = cv_model(lambda: MLPClassifier(hidden_layer_sizes=(128, 64, 32), alpha=1e-3,
                    batch_size=128, learning_rate='adaptive', max_iter=200,
                    early_stopping=True, random_state=SEED),
                    tab_all, y_all, seasons_all, "Sklearn-MLP", use_scaled=True)
all_oof['SkMLP'], all_brier['SkMLP'] = oof, bs

# 10. XGBoost
if HAS_XGB:
    oof, bs = cv_model(lambda: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                        gamma=0.2, reg_alpha=0.1, reg_lambda=1.0,
                        objective='binary:logistic', tree_method='hist',
                        random_state=SEED, verbosity=0),
                        tab_all, y_all, seasons_all, "XGBoost")
    all_oof['XGB'], all_brier['XGB'] = oof, bs

# 11. LightGBM
if HAS_LGB:
    oof, bs = cv_model(lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                        min_child_samples=20, verbose=-1, random_state=SEED),
                        tab_all, y_all, seasons_all, "LightGBM")
    all_oof['LGB'], all_brier['LGB'] = oof, bs

# 12. CatBoost
if HAS_CB:
    oof, bs = cv_model(lambda: CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5,
                        l2_leaf_reg=3.0, random_seed=SEED, verbose=0),
                        tab_all, y_all, seasons_all, "CatBoost")
    all_oof['CB'], all_brier['CB'] = oof, bs

# 13. Seed Prior (analytical baseline)
print("\n  Building Seed Prior...")
oof_seed = np.full(len(y_all), np.nan)
for i in range(len(y_all)):
    row = meta_all.iloc[i]
    diff = row['SeedA'] - row['SeedB']
    oof_seed[i] = np.clip(1.0 / (1.0 + 10.0 ** (diff * 0.15)), CLIP_MIN, CLIP_MAX)
bs_seed = np.mean((y_all - oof_seed) ** 2)
print(f"  {'SeedPrior':30s}: Brier={bs_seed:.4f}")
all_oof['SeedPrior'], all_brier['SeedPrior'] = oof_seed, bs_seed

print(f"\nTotal models: {len(all_oof)}")

## 4. Mega-Ensemble Optimization

In [ ]:
print("\n" + "=" * 60)
print("MEGA-ENSEMBLE OPTIMIZATION")
print("=" * 60)

# Common valid mask
common_valid = np.ones(len(y_all), dtype=bool)
for name, oof in all_oof.items():
    common_valid &= ~np.isnan(oof)
print(f"Common valid: {common_valid.sum()} samples")
y_valid = y_all[common_valid]

# Rankings
print("\nModel Rankings (Brier Score):")
print("-" * 50)
for rank, (name, bs) in enumerate(sorted(all_brier.items(), key=lambda x: x[1]), 1):
    bar = '|' * int((0.25 - bs) * 200)
    print(f"  {rank:2d}. {name:25s}: {bs:.4f} {bar}")

In [ ]:
# Optimize weights
names = list(all_oof.keys())
preds_matrix = np.column_stack([all_oof[n][common_valid] for n in names])

def mega_objective(weights):
    w = weights / weights.sum()
    ens = np.clip(preds_matrix @ w, CLIP_MIN, CLIP_MAX)
    return np.mean((y_valid - ens) ** 2)

n_models = len(names)
x0 = np.ones(n_models) / n_models
bounds = [(0, 1)] * n_models
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1.0}
result = minimize(mega_objective, x0, bounds=bounds, constraints=constraints, method='SLSQP')

mega_weights = dict(zip(names, result.x))
mega_brier = result.fun

print(f"\nMEGA-ENSEMBLE Brier: {mega_brier:.4f}")

# Simple average
simple_ens = np.clip(preds_matrix.mean(axis=1), CLIP_MIN, CLIP_MAX)
simple_brier = np.mean((y_valid - simple_ens) ** 2)
print(f"Simple Average Brier: {simple_brier:.4f}")

print("\nOptimal Weights (non-zero):")
for name, w in sorted(mega_weights.items(), key=lambda x: -x[1]):
    if w > 0.005:
        print(f"  {name:25s}: {w:.3f} ({w:.1%})")

### 4.1 Stacking Meta-Learner

In [ ]:
print("\nStacking Meta-Learner:")
meta_X = preds_matrix
meta_y = y_valid
meta_seasons = seasons_all[common_valid]
meta_oof = np.full(len(meta_y), np.nan)

for vs in sorted(set(s for s in np.unique(meta_seasons) if s >= 2018)):
    tr = meta_seasons < vs
    va = meta_seasons == vs
    if va.sum() == 0: continue
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(meta_X[tr], meta_y[tr])
    meta_oof[va] = np.clip(meta_model.predict(meta_X[va]), CLIP_MIN, CLIP_MAX)

meta_valid = ~np.isnan(meta_oof)
if meta_valid.sum() > 0:
    stacking_brier = np.mean((meta_y[meta_valid] - meta_oof[meta_valid]) ** 2)
    print(f"  Stacking Brier: {stacking_brier:.4f}")
else:
    stacking_brier = mega_brier

# Pick best approach
best_brier = min(mega_brier, stacking_brier)
use_stacking = stacking_brier < mega_brier
print(f"\nBest Approach: {'Stacking' if use_stacking else 'Weighted Ensemble'} (Brier={best_brier:.4f})")

### 4.2 Calibration Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Top 5 individual models
top5 = sorted(all_brier.items(), key=lambda x: x[1])[:5]
for name, _ in top5:
    p = all_oof[name][common_valid]
    frac, mean_p = calibration_curve(y_valid, p, n_bins=10)
    axes[0].plot(mean_p, frac, 's-', label=name, markersize=4)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('Top 5 Individual Models'); axes[0].legend(fontsize=7)

# Mega ensemble
w_arr = np.array([mega_weights[n] for n in names])
mega_pred = np.clip(preds_matrix @ w_arr, CLIP_MIN, CLIP_MAX)
frac, mean_p = calibration_curve(y_valid, mega_pred, n_bins=10)
axes[1].plot(mean_p, frac, 's-', label='Mega-Ensemble', linewidth=2, color='red')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_title('Mega-Ensemble Calibration'); axes[1].legend()

# Prediction distribution
axes[2].hist(mega_pred, bins=50, alpha=0.7, label='Ensemble', color='steelblue', edgecolor='white')
axes[2].axvline(0.5, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Prediction Distribution'); axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'mega_ensemble_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Inference & Final Submission

In [ ]:
print("\n" + "=" * 60)
print("FINAL INFERENCE PIPELINE")
print("=" * 60)

# Retrain ALL models on full data
print("[1/3] Retraining all models on full data...")

final_models = {}

# LR variants
for c_val in [0.1, 0.5, 1.0]:
    key = f'LR-{c_val}'
    if key in all_oof:
        m = LogisticRegression(C=c_val, max_iter=1000, random_state=SEED)
        m.fit(tab_scaled, y_all)
        final_models[key] = ('scaled', m)

# Elastic Net
if 'ElasticNet' in all_oof:
    m = SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-4, l1_ratio=0.5,
                       max_iter=1000, random_state=SEED)
    m.fit(tab_scaled, y_all)
    final_models['ElasticNet'] = ('scaled', m)

# Bayesian Ridge
if 'BayesianRidge' in all_oof:
    m = BayesianRidge(max_iter=300)
    m.fit(tab_scaled, y_all)
    final_models['BayesianRidge'] = ('scaled_reg', m)

# SVM
if 'SVM-RBF' in all_oof:
    m = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=SEED)
    m.fit(tab_scaled, y_all)
    final_models['SVM-RBF'] = ('scaled', m)

# KNN
if 'KNN' in all_oof:
    m = KNeighborsClassifier(n_neighbors=50, weights='distance')
    m.fit(tab_scaled, y_all)
    final_models['KNN'] = ('scaled', m)

# RF
if 'RF' in all_oof:
    m = RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=10,
                                 max_features='sqrt', random_state=SEED, n_jobs=-1)
    m.fit(tab_all, y_all)
    final_models['RF'] = ('raw', m)

# ET
if 'ET' in all_oof:
    m = ExtraTreesClassifier(n_estimators=500, max_depth=10, min_samples_leaf=8,
                               max_features='sqrt', random_state=SEED, n_jobs=-1)
    m.fit(tab_all, y_all)
    final_models['ET'] = ('raw', m)

# GBC
if 'GBC' in all_oof:
    m = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                     subsample=0.8, min_samples_leaf=10, random_state=SEED)
    m.fit(tab_all, y_all)
    final_models['GBC'] = ('raw', m)

# Sklearn MLP
if 'SkMLP' in all_oof:
    m = MLPClassifier(hidden_layer_sizes=(128, 64, 32), alpha=1e-3, batch_size=128,
                       learning_rate='adaptive', max_iter=200, random_state=SEED)
    m.fit(tab_scaled, y_all)
    final_models['SkMLP'] = ('scaled', m)

# XGBoost
if HAS_XGB and 'XGB' in all_oof:
    m = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                        gamma=0.2, reg_alpha=0.1, reg_lambda=1.0,
                        objective='binary:logistic', tree_method='hist',
                        random_state=SEED, verbosity=0)
    m.fit(tab_all, y_all)
    final_models['XGB'] = ('raw', m)

# LightGBM
if HAS_LGB and 'LGB' in all_oof:
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                          num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                          min_child_samples=20, verbose=-1, random_state=SEED)
    m.fit(tab_all, y_all)
    final_models['LGB'] = ('raw', m)

# CatBoost
if HAS_CB and 'CB' in all_oof:
    m = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=5,
                              l2_leaf_reg=3.0, random_seed=SEED, verbose=0)
    m.fit(tab_all, y_all)
    final_models['CB'] = ('raw', m)

# Stacking meta-learner
meta_model_final = Ridge(alpha=1.0)
meta_model_final.fit(meta_X, meta_y)

print(f"  Retrained {len(final_models)} models.")

In [ ]:
# ========== PREDICT FUNCTION ==========
all_stats_combined = pd.concat([m_stats, w_stats], ignore_index=True)
all_elo_combined = pd.concat([m_elo_df, w_elo_df], ignore_index=True)

def predict_matchup_mega(season, team_a, team_b):
    """Ultimate mega-ensemble prediction."""
    vec_a = get_team_vector(all_stats_combined, all_elo_combined, season, team_a)
    vec_b = get_team_vector(all_stats_combined, all_elo_combined, season, team_b)
    if vec_a is None: vec_a = np.zeros(N_TEAM_FEATURES, dtype=np.float32)
    if vec_b is None: vec_b = np.zeros(N_TEAM_FEATURES, dtype=np.float32)

    sa = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == team_a)]
    sb = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == team_b)]
    seed_a = sa.iloc[0]['SeedNum'] if len(sa) > 0 else 8
    seed_b = sb.iloc[0]['SeedNum'] if len(sb) > 0 else 8

    diff = vec_a - vec_b
    tab = list(diff) + [seed_a - seed_b, seed_a, seed_b]

    # Massey
    is_mens = 1000 <= team_a <= 1999
    if is_mens:
        am = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_a)]
        bm = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_b)]
        for sys_name in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
            if len(am) > 0 and len(bm) > 0 and sys_name in am.columns:
                va = am.iloc[0][sys_name] if not pd.isna(am.iloc[0].get(sys_name)) else 150
                vb = bm.iloc[0][sys_name] if not pd.isna(bm.iloc[0].get(sys_name)) else 150
                tab.append(va - vb)
            else:
                tab.append(0)
    else:
        tab.extend([0, 0, 0, 0])

    # Interactions
    elo_diff = vec_a[-1] - vec_b[-1]
    net_idx = TEAM_FEATURES.index('NetRating')
    net_diff = diff[net_idx]
    seed_diff = seed_a - seed_b
    tab.extend([seed_diff * elo_diff, seed_diff * net_diff])

    tab = np.array(tab, dtype=np.float32)
    if len(tab) < N_TAB:
        tab = np.pad(tab, (0, N_TAB - len(tab)))
    tab = tab[:N_TAB]
    tab = np.nan_to_num(tab, nan=0.0).reshape(1, -1)
    tab_s = tab_scaler.transform(tab)

    preds = {}
    for name, (mode, model) in final_models.items():
        X = tab_s if mode == 'scaled' else tab
        if mode == 'scaled_reg':
            preds[name] = np.clip(model.predict(X)[0], CLIP_MIN, CLIP_MAX)
        elif hasattr(model, 'predict_proba'):
            preds[name] = model.predict_proba(X)[0, 1]
        else:
            preds[name] = np.clip(model.predict(X)[0], CLIP_MIN, CLIP_MAX)

    # Seed prior
    preds['SeedPrior'] = 1.0 / (1.0 + 10.0 ** (seed_diff * 0.15))

    # Weighted ensemble
    final_pred = sum(mega_weights.get(n, 0) * np.clip(p, CLIP_MIN, CLIP_MAX) for n, p in preds.items())
    total_w = sum(mega_weights.get(n, 0) for n in preds if n in mega_weights)
    if total_w > 0:
        final_pred /= total_w

    return np.clip(final_pred, CLIP_MIN, CLIP_MAX)

In [ ]:
print("\n[2/3] Generating submissions...")

def generate_submission(sub_df, filename):
    predictions = []
    for i, row in sub_df.iterrows():
        parts = row['ID'].split('_')
        season, ta, tb = int(parts[0]), int(parts[1]), int(parts[2])
        pred = predict_matchup_mega(season, ta, tb)
        predictions.append(pred)
        if (i + 1) % 50000 == 0:
            print(f"    {i+1}/{len(sub_df)} predictions done...")
    sub_df = sub_df.copy()
    sub_df['Pred'] = predictions
    sub_df.to_csv(OUT_DIR / filename, index=False)
    preds_arr = np.array(predictions)
    print(f"  {filename}: {len(sub_df)} rows, mean={preds_arr.mean():.4f}, "
          f"std={preds_arr.std():.4f}, range=[{preds_arr.min():.4f}, {preds_arr.max():.4f}]")
    return sub_df

sub1_mega = generate_submission(sub1, 'submission_stage1_mega.csv')
sub2_mega = generate_submission(sub2, 'submission_stage2_mega.csv')

# Conservative blend
print("\n[3/3] Creating conservative submission...")
seed_preds = []
for _, row in sub2.iterrows():
    parts = row['ID'].split('_')
    ta, tb, s = int(parts[1]), int(parts[2]), int(parts[0])
    sa = all_seeds[(all_seeds['Season'] == s) & (all_seeds['TeamID'] == ta)]
    sb = all_seeds[(all_seeds['Season'] == s) & (all_seeds['TeamID'] == tb)]
    if len(sa) > 0 and len(sb) > 0:
        diff = sa.iloc[0]['SeedNum'] - sb.iloc[0]['SeedNum']
        seed_preds.append(1.0 / (1.0 + 10.0 ** (diff * 0.15)))
    else:
        seed_preds.append(0.5)
seed_preds = np.array(seed_preds)

model_preds = sub2_mega['Pred'].values
conservative = np.clip(0.15 * seed_preds + 0.85 * model_preds, CLIP_MIN, CLIP_MAX)
sub2_cons = sub2.copy()
sub2_cons['Pred'] = conservative
sub2_cons.to_csv(OUT_DIR / 'submission_stage2_mega_conservative.csv', index=False)
print(f"  Conservative: mean={conservative.mean():.4f}")

## 6. Final Summary

In [ ]:
print("\n" + "=" * 60)
print("MEGA-ENSEMBLE - ULTIMATE RESULTS")
print("=" * 60)

print(f"\nTotal Models in Ensemble: {len(all_oof)}")
print(f"\nModel Rankings (Brier Score):")
print("-" * 55)
for rank, (name, bs) in enumerate(sorted(all_brier.items(), key=lambda x: x[1]), 1):
    bar = '|' * int((0.25 - bs) * 200)
    print(f"  {rank:2d}. {name:22s}: {bs:.4f} {bar}")

print(f"\n  {'MEGA-ENSEMBLE':22s}: {mega_brier:.4f} ***")
print(f"  {'Stacking':22s}: {stacking_brier:.4f}")
print(f"  {'Simple Average':22s}: {simple_brier:.4f}")
print(f"  {'Best Single Model':22s}: {min(all_brier.values()):.4f}")

improvement = min(all_brier.values()) - mega_brier
print(f"\n  Ensemble improves over best single by: {improvement:.4f} ({improvement/min(all_brier.values())*100:.1f}%)")

print(f"\nEnsemble Weights (non-zero):")
for name, w in sorted(mega_weights.items(), key=lambda x: -x[1]):
    if w > 0.005:
        print(f"  {name:22s}: {w:.1%}")

print(f"\nSubmissions:")
print(f"  {OUT_DIR / 'submission_stage1_mega.csv'} (stage 1 validation)")
print(f"  {OUT_DIR / 'submission_stage2_mega.csv'} (stage 2 aggressive)")
print(f"  {OUT_DIR / 'submission_stage2_mega_conservative.csv'} (stage 2 conservative)")

print(f"\n{'='*60}")
print("RECOMMENDED STRATEGY:")
print("  1. Submit BOTH mega aggressive AND conservative")
print("  2. Select the one with better Stage 1 score for final")
print("  3. Keep the other as backup selection")
print(f"{'='*60}")